# Minimal: OpenAI chat + Sentence-Transformers embeddings

<sup>Part of the DGT Summer School 2026 material.</sup>

---

The smallest possible example of the two building blocks used throughout the course:

1. **Chat completions** via the official `openai` library.
2. **Text embeddings** computed **locally** with `sentence-transformers` (CPU-only, no API).

All three connection settings are read from the `.env` file so nothing is hard-coded:

| Variable | Meaning |
|---|---|
| `OPENAI_API_KEY` | Your API key. |
| `OPENAI_MODEL` | Chat model id (e.g. `gpt-4o-mini`). |
| `OPENAI_BASE_URL` | API endpoint — change it to point at a proxy or a local/OpenAI-compatible server. |

In [2]:
# Load the configurable settings from the local .env file (never committed).
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")

assert API_KEY, "OPENAI_API_KEY not found - add it to your .env file."
print(f"Model    : {MODEL}")
print(f"Base URL : {BASE_URL}")

Model    : gpt-4o-mini
Base URL : https://api.openai.com/v1


## 1  Chat completion with the `openai` library

We create one `OpenAI` client, passing the key and base URL explicitly so the same code works against OpenAI or any OpenAI-compatible endpoint.

In [3]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In one sentence, what is an embedding?"},
    ],
)

print(response.choices[0].message.content)

An embedding is a dense vector representation of an object, such as words or items, that captures semantic relationships in a high-dimensional space.


## 2  Local embeddings with `sentence-transformers`

Embeddings turn text into vectors so we can measure semantic similarity. Here we compute them **locally on the CPU** — no API call, no key needed. The first run downloads the (small) model; later runs use the cache.

In [4]:
from sentence_transformers import SentenceTransformer

# A small, fast, general-purpose model (~80 MB) that runs comfortably on CPU.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "A cat sits on the mat.",
    "A kitten rests on the rug.",
    "The stock market fell sharply today.",
]

embeddings = embedder.encode(sentences)
print(f"Shape: {embeddings.shape}  (one {embeddings.shape[1]}-dim vector per sentence)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape: (3, 384)  (one 384-dim vector per sentence)


In [5]:
# Cosine similarity: the two cat/kitten sentences should be far more similar
# to each other than to the finance sentence.
from sentence_transformers import util

sim = util.cos_sim(embeddings, embeddings)

for i, a in enumerate(sentences):
    for j, b in enumerate(sentences):
        if i < j:
            print(f"{sim[i][j]:.3f}  |  {a!r}  <->  {b!r}")

0.666  |  'A cat sits on the mat.'  <->  'A kitten rests on the rug.'
0.091  |  'A cat sits on the mat.'  <->  'The stock market fell sharply today.'
0.105  |  'A kitten rests on the rug.'  <->  'The stock market fell sharply today.'
